# SSH Web Tool - Jupyter Notebook 集成示例

本示例展示如何在 Jupyter Notebook 中通过 **SSHClient** 复用已有的 SSH Web Tool 服务。

## 前置条件
1. 已经启动 SSH Web Tool 服务：`python main.py`（或 `uv run python main.py`）
2. 服务地址：http://127.0.0.1:8765
3. 在浏览器中打开 http://127.0.0.1:8765 可以观察所有 SSH 终端

## 核心概念
- **SSHClient**：通过 HTTP API 与后端通信，复用后端的 SSH 会话池
- 所有在 Notebook 中创建的 SSH 连接和执行的命令都会实时显示在 Web UI 中
- 连接状态由后端统一维护，Notebook 关闭后连接不会中断

In [1]:
from ssh_client import SSHClient

# 连接到已有的后端服务，不启动新的 Web UI
# 确保你已经通过 `python main.py` 启动了服务
client = SSHClient("http://127.0.0.1:8765")

print('SSHClient 已初始化')
print('服务地址: http://127.0.0.1:8765')
print('在浏览器中打开上面的地址，可以观察所有 SSH 终端')

SSHClient 已初始化
服务地址: http://127.0.0.1:8765
在浏览器中打开上面的地址，可以观察所有 SSH 终端


In [2]:
# SSH 连接配置（请修改为你的实际服务器）
SSH_HOST = '47.108.145.84'      # 服务器地址
SSH_PORT = 22                     # 端口
SSH_USER = 'root'                 # 用户名
SSH_PASS = 'wrz@1234'             # 密码

print(f'目标服务器: {SSH_USER}@{SSH_HOST}:{SSH_PORT}')

目标服务器: root@47.108.145.84:22


In [3]:
# 建立 SSH 连接（会自动启动交互式终端）
# 这个连接会实时显示在 Web UI 中
session_id = client.get_or_create_session(
    host=SSH_HOST,
    port=SSH_PORT,
    username=SSH_USER,
    password=SSH_PASS,
)

print(f'SSH 连接成功！')
print(f'会话 ID: {session_id}')
print()
print('现在请在浏览器中打开 http://127.0.0.1:8765')
print('你应该能看到一个新的终端标签')
print('后续在 Notebook 中执行的命令都会实时显示在这个终端中')

SSH 连接成功！
会话 ID: 7ef91688

现在请在浏览器中打开 http://127.0.0.1:8765
你应该能看到一个新的终端标签
后续在 Notebook 中执行的命令都会实时显示在这个终端中


In [4]:
# 发送第一个命令：查看系统信息
# 命令会注入到交互式终端，Web UI 中可见，CLI 也能拿到输出
result = client.run_command(session_id, 'uname -a && uptime')

print('=== 命令执行结果 ===')
print(result.get('stdout', ''))
print(f'返回码: {result.get("returncode", "N/A")}')
print()
print('注意：在 Web UI 终端中应该能看到这条命令被输入和执行的过程！')

=== 命令执行结果 ===
Linux iZ2vcj68n53f7yrz7sumeiZ 7.0.0-30-generic #30-Ubuntu SMP PREEMPT_DYNAMIC Fri Jul 31 18:22:54 UTC 2026 x86_64 GNU/Linux
 00:21:43 up 1 day, 36 min,  1 user,  load average: 0.15, 0.12, 0.10

返回码: 0

注意：在 Web UI 终端中应该能看到这条命令被输入和执行的过程！


In [5]:
# 发送第二个命令：查看当前目录
result = client.run_command(session_id, 'pwd && ls -la')

print('=== 当前目录和文件 ===')
print(result.get('stdout', ''))

=== 当前目录和文件 ===
/root
total 60
drwx------  7 root root 4096 Sep  6 18:25 .
drwxr-xr-x 20 root root 4096 Sep  5 23:45 ..
-rw-------  1 root root   69 Sep  6 18:25 .Xauthority
-rw-------  1 root root 2819 Sep  7 00:21 .bash_history
-rw-r--r--  1 root root 3106 Apr 20 16:46 .bashrc
drwx------  2 root root 4096 Aug 28 09:50 .cache
drwx------  3 root root 4096 Sep  5 23:52 .config
drwxr-xr-x  2 root root 4096 Aug 28 10:01 .pip
-rw-r--r--  1 root root  132 Apr 20 16:46 .profile
-rw-r--r--  1 root root   72 Aug 28 10:01 .pydistutils.cfg
-rw-r--r--  1 root root  280 Sep  7 00:16 .python_history
drwx------  2 root root 4096 Aug 28 09:50 .ssh
-rw-r--r--  1 root root 5039 Sep  6 00:23 sessions.py
drwxr-xr-x  2 root root 4096 Sep  5 23:45 workspace



In [6]:
# 连续执行：切换目录后查看文件
# 因为是在同一个交互式终端中执行，所以 cd 等状态会保持
client.run_command(session_id, 'cd /tmp')
result = client.run_command(session_id, 'pwd')
print(f'切换后目录: {result.get("stdout", "").strip()}')  # 应该是 /tmp

# 再切回根目录
client.run_command(session_id, 'cd /')
result = client.run_command(session_id, 'pwd')
print(f'切回后目录: {result.get("stdout", "").strip()}')  # 应该是 /

切换后目录: /root
切回后目录: /root


In [7]:
# 进入 Python 交互模式
# 注意：进入交互程序后，命令执行结果的检测可能不太准确
# 建议在 Web UI 中直接观察输出
result = client.run_command(session_id, 'python3')

print('=== 进入 Python ===')
print(result.get('stdout', ''))
print()
print('在 Web UI 终端中应该能看到 Python 提示符 >>>')

TimeoutError: timed out

In [ ]:
# 在 Python 中执行代码
# 注意：在 Python 交互模式下，输出捕获可能不太准确
# 建议主要在 Web UI 中观察输出
result = client.run_command(session_id, 'print("Hello from Jupyter!")')

print('=== Python 执行结果 ===')
print(result.get('stdout', ''))

# 计算 1+1
result = client.run_command(session_id, '1 + 1')
print('=== 1 + 1 = ===')
print(result.get('stdout', ''))

In [ ]:
# 退出 Python，回到 shell
result = client.run_command(session_id, 'exit()')

print('=== 退出 Python ===')
print(result.get('stdout', ''))
print()
print('在 Web UI 终端中应该回到 shell 提示符')

In [ ]:
# 查看终端当前状态
state = client.get_state(session_id)

print('=== 终端状态 ===')
print(f'提示符类型: {state.get("prompt_type", "unknown")}')
print(f'是否忙碌: {state.get("is_busy", False)}')
print(f'最后输出: {state.get("last_output", "")[:100]}')

In [ ]:
# 列出所有活跃会话
# 包括 Web UI、CLI、SDK、Notebook 创建的所有会话
sessions = client.list_sessions()

print(f'当前活跃会话数: {len(sessions)}')
print()
for s in sessions:
    print(f'  会话ID: {s.get("session_id", "N/A")}')
    print(f'  主机: {s.get("host", "N/A")}:{s.get("port", 22)}')
    print(f'  用户: {s.get("username", "N/A")}')
    print(f'  终端名: {s.get("terminal_name", "N/A")}')
    print(f'  已连接: {s.get("connected", False)}')
    print(f'  有shell: {s.get("has_shell", False)}')
    print()

In [ ]:
# 注意：SSHClient 目前没有直接的 send_input 方法
# 发送控制字符可以通过 run_command 执行特殊命令，或者在 Web UI 中直接操作

print('控制字符发送方式：')
print('1. 在 Web UI 终端中直接操作')
print('2. 通过 run_command 执行命令')
print('3. Ctrl+C = 中断当前命令（在 Web UI 中按 Ctrl+C）')
print('4. Ctrl+D = 退出当前程序（在 Web UI 中按 Ctrl+D）')

In [ ]:
# 关闭单个会话
# client.close_session(session_id)
# print(f'会话 {session_id} 已关闭')

# 注意：关闭会话后，Web UI 中的终端标签也会消失
# 如果只是想关闭 Notebook 而保留连接，不需要调用 close_session
# 连接状态由后端统一维护，Notebook 关闭后连接不会中断

print('取消注释上面的代码来关闭连接')
print('保持连接打开的话，可以继续在 Web UI 中操作终端')

## 进阶：使用 exec_host 直接用 IP 执行命令

如果你不想手动管理 session_id，可以使用 `exec_host` 方法，直接用主机 ID 或 IP 执行命令，它会自动复用该主机的第一个活跃终端。

In [ ]:
# 使用 exec_host 直接用 IP 执行命令
# 自动复用该主机的第一个活跃终端，不需要手动管理 session_id
result = client.exec_host('47.108.145.84', 'echo "Hello from exec_host!" && date')

print('=== exec_host 执行结果 ===')
print(result.get('stdout', ''))
print(f'返回码: {result.get("returncode", "N/A")}')

# exec_host 的智能 ID 识别优先级：
# 1. session_id - 如果匹配到活跃终端的会话 ID，直接使用该终端
# 2. host_id - 如果匹配到保存主机的 ID，复用该主机的第1个活跃终端
# 3. IP 地址 - 如果是 ip 或 ip:port 格式，按 IP 查找主机
# 4. 都不匹配 - 按 host_id 处理，没有则新建连接